# Joins

Notebook version of `joins.py` so each step can be rerun independently while working through the lesson.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Joins Notebook")
    .config("spark.master", "local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

26/03/19 21:51:41 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

In [2]:
guitars_df = spark.read.json("src/main/resources/data/guitars.json")
guitarists_df = spark.read.json("src/main/resources/data/guitarPlayers.json")
bands_df = spark.read.json("src/main/resources/data/bands.json")

guitars_df.show(truncate=False)
guitarists_df.show(truncate=False)
bands_df.show(truncate=False)

+----------------------+---+------+------------+
|guitarType            |id |make  |model       |
+----------------------+---+------+------------+
|Electric double-necked|0  |Gibson|EDS-1275    |
|Electric              |5  |Fender|Stratocaster|
|Electric              |1  |Gibson|SG          |
|Acoustic              |2  |Taylor|914         |
|Electric              |3  |ESP   |M-II        |
+----------------------+---+------+------------+

+----+-------+---+------------+
|band|guitars|id |name        |
+----+-------+---+------------+
|0   |[0]    |0  |Jimmy Page  |
|1   |[1]    |1  |Angus Young |
|2   |[1, 5] |2  |Eric Clapton|
|3   |[3]    |3  |Kirk Hammett|
+----+-------+---+------------+

+-----------+---+------------+----+
|hometown   |id |name        |year|
+-----------+---+------------+----+
|Sydney     |1  |AC/DC       |1973|
|London     |0  |Led Zeppelin|1968|
|Los Angeles|3  |Metallica   |1981|
|Liverpool  |4  |The Beatles |1960|
+-----------+---+------------+----+



In [3]:
# Inner join: only matching rows from both sides are kept.
join_condition = guitarists_df["band"] == bands_df["id"]
guitarists_bands_df = guitarists_df.join(bands_df, join_condition, "inner")

guitarists_bands_df.show()

+----+-------+---+------------+-----------+---+------------+----+
|band|guitars| id|        name|   hometown| id|        name|year|
+----+-------+---+------------+-----------+---+------------+----+
|   1|    [1]|  1| Angus Young|     Sydney|  1|       AC/DC|1973|
|   0|    [0]|  0|  Jimmy Page|     London|  0|Led Zeppelin|1968|
|   3|    [3]|  3|Kirk Hammett|Los Angeles|  3|   Metallica|1981|
+----+-------+---+------------+-----------+---+------------+----+



In [4]:
# Left outer join: inner join and LEFT table
# # keep everything from the guitarists side, fill missing band data with nulls.
# Includes eric clapton when he has no band
guitarists_df.join(bands_df, join_condition, "left_outer").show()

+----+-------+---+------------+-----------+----+------------+----+
|band|guitars| id|        name|   hometown|  id|        name|year|
+----+-------+---+------------+-----------+----+------------+----+
|   0|    [0]|  0|  Jimmy Page|     London|   0|Led Zeppelin|1968|
|   1|    [1]|  1| Angus Young|     Sydney|   1|       AC/DC|1973|
|   2| [1, 5]|  2|Eric Clapton|       NULL|NULL|        NULL|NULL|
|   3|    [3]|  3|Kirk Hammett|Los Angeles|   3|   Metallica|1981|
+----+-------+---+------------+-----------+----+------------+----+



In [5]:
# right outer join - everything in inner join + everything in RIGHT table
# should include beatles that has no guitarist
guitarists_df.join(bands_df, join_condition, "right_outer").show()

+----+-------+----+------------+-----------+---+------------+----+
|band|guitars|  id|        name|   hometown| id|        name|year|
+----+-------+----+------------+-----------+---+------------+----+
|   1|    [1]|   1| Angus Young|     Sydney|  1|       AC/DC|1973|
|   0|    [0]|   0|  Jimmy Page|     London|  0|Led Zeppelin|1968|
|   3|    [3]|   3|Kirk Hammett|Los Angeles|  3|   Metallica|1981|
|NULL|   NULL|NULL|        NULL|  Liverpool|  4| The Beatles|1960|
+----+-------+----+------------+-----------+---+------------+----+



In [6]:
# full outer join = everything in inner and BOTH tables
# will have eric clapton and the beatles
guitarists_df.join(bands_df, join_condition, "outer").show()

+----+-------+----+------------+-----------+----+------------+----+
|band|guitars|  id|        name|   hometown|  id|        name|year|
+----+-------+----+------------+-----------+----+------------+----+
|   0|    [0]|   0|  Jimmy Page|     London|   0|Led Zeppelin|1968|
|   1|    [1]|   1| Angus Young|     Sydney|   1|       AC/DC|1973|
|   2| [1, 5]|   2|Eric Clapton|       NULL|NULL|        NULL|NULL|
|   3|    [3]|   3|Kirk Hammett|Los Angeles|   3|   Metallica|1981|
|NULL|   NULL|NULL|        NULL|  Liverpool|   4| The Beatles|1960|
+----+-------+----+------------+-----------+----+------------+----+



In [7]:
# semi-joins
# only show rows from left table that has data in right table, but does not include right table data
# returns guitarists that have bands but no band data
guitarists_df.join(bands_df, join_condition, "left_semi").show()

+----+-------+---+------------+
|band|guitars| id|        name|
+----+-------+---+------------+
|   0|    [0]|  0|  Jimmy Page|
|   1|    [1]|  1| Angus Young|
|   3|    [3]|  3|Kirk Hammett|
+----+-------+---+------------+



In [8]:
# left anti join
# only show rows from left table that DO NOT have data in right table, but does not include right table data
# returns eric clapton since he does not have a banda
guitarists_df.join(bands_df, join_condition, "left_anti").show()

+----+-------+---+------------+
|band|guitars| id|        name|
+----+-------+---+------------+
|   2| [1, 5]|  2|Eric Clapton|
+----+-------+---+------------+



In [ ]:
# things to bear in mind when doing joins
# this will crash because id is now ambigous after join because they both had id col
# guitarists_bands_df.select("id", "band").show

In [ ]:
# option 1 - rename the column on which we are joining
# if we rename the conflicting cols we wont have issues
guitarists_df.join(bands_df.withColumnRenamed("id", "band"), "band")

In [ ]:
# option 2 - drop the dupe column
# notice that we are doing the drop after join, this is allowed because
# spark remembers where we got the columns it knows bands df gave us id column after join
# spark keeps unique id for where columns came from
guitarists_bands_df.drop(bands_df.col("id"))

In [10]:
# option 3 - rename the offending column and keep the data
# less attractive option because we still have duplicate data band and band id hold same data (duplicate)
bandsModDF = bands_df.withColumnRenamed("id", "bandId")
guitarists_df.join(bandsModDF, guitarists_df["band"] == bandsModDF["bandId"]).show()

+----+-------+---+------------+-----------+------+------------+----+
|band|guitars| id|        name|   hometown|bandId|        name|year|
+----+-------+---+------------+-----------+------+------------+----+
|   1|    [1]|  1| Angus Young|     Sydney|     1|       AC/DC|1973|
|   0|    [0]|  0|  Jimmy Page|     London|     0|Led Zeppelin|1968|
|   3|    [3]|  3|Kirk Hammett|Los Angeles|     3|   Metallica|1981|
+----+-------+---+------------+-----------+------+------------+----+



In [12]:
from pyspark.sql.functions import expr

# using complex types in joins
# guitars is an array in guitarists since guitarists have many guitars
# notice resulting table will have multiple rows for each guitarists for each guitar they own
guitarists_df.join(guitars_df.withColumnRenamed("id", "guitarId"), expr("array_contains(guitars, guitarId)")).show()


+----+-------+---+------------+--------------------+--------+------+------------+
|band|guitars| id|        name|          guitarType|guitarId|  make|       model|
+----+-------+---+------------+--------------------+--------+------+------------+
|   0|    [0]|  0|  Jimmy Page|Electric double-n...|       0|Gibson|    EDS-1275|
|   2| [1, 5]|  2|Eric Clapton|            Electric|       5|Fender|Stratocaster|
|   1|    [1]|  1| Angus Young|            Electric|       1|Gibson|          SG|
|   2| [1, 5]|  2|Eric Clapton|            Electric|       1|Gibson|          SG|
|   3|    [3]|  3|Kirk Hammett|            Electric|       3|   ESP|        M-II|
+----+-------+---+------------+--------------------+--------+------+------------+



In [ ]:
# Exercises
# 1. show all employees and their max salary they had
# 2. show all employees who were never managers, keyword is NEVER so no need to worry about dates
# 3. find job titles of the top 10 best employees of the company, max(toDate) is current job title

# Tips - try writing sql first then transfrom it to dataframe constructs
# - try to do intermediate dataframes and preview then step by step 

In [ ]:
# 1. show all employees and their max salary they had
# sub steps
# 1.1 pull in employees df 
# 1.2 pull in salaries df 
# 1.3 write how sql will look like
# 1.4 take max salary per employee
# 1.5 join with employees table

In [18]:
# 1.1 + 1.2
# pull in data 
import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.functions import col, count, count_distinct, approx_count_distinct, avg, stddev, mean, min, sum


employeesDF = ( 
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("user", "docker")
    .option("password", "docker")
    .option("dbtable", "public.employees")
    .option("url", "jdbc:postgresql://postgres:5432/rtjvm")
    .load()
)

salariesDF = ( 
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("user", "docker")
    .option("password", "docker")
    .option("dbtable", "public.salaries")
    .option("url", "jdbc:postgresql://postgres:5432/rtjvm")
    .load()
)

employeesDF.printSchema()
salariesDF.printSchema()

root
 |-- emp_no: integer (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- hire_date: date (nullable = true)

root
 |-- emp_no: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- from_date: date (nullable = true)
 |-- to_date: date (nullable = true)



In [19]:
# 1.3 write how sql will look like
# WITH max_salary_by_employee AS (
#     SELECT emp_no, MAX(salary) AS max_salary
#     FROM salaries
#     GROUP BY emp_no
# )
# SELECT *
# FROM employees e
# JOIN max_salary_by_employee s
#   ON s.emp_no = e.emp_no;

In [26]:
# 1.4 take max salary per employee
# group salaries by employee
# take sum on that group of salary column


maxSalaryByEmployee = (
    salariesDF.select(col("emp_no"), col("salary"))
    .groupBy(col("emp_no"))
    .max("salary")
)

maxSalaryByEmployee.show(5)

+------+-----------+
|emp_no|max(salary)|
+------+-----------+
| 12940|      85425|
| 13840|      41453|
| 14450|      75524|
| 14570|      72506|
| 15790|      79009|
+------+-----------+
only showing top 5 rows



In [31]:
# 1.5 join with employees table
employeeWithMaxSalaryDF = employeesDF.join(maxSalaryByEmployee, "emp_no", "left")

employeeWithMaxSalaryDF.show(5)

+------+----------+----------+---------+------+----------+-----------+
|emp_no|birth_date|first_name|last_name|gender| hire_date|max(salary)|
+------+----------+----------+---------+------+----------+-----------+
| 10020|1952-12-24|    Mayuko|  Warwick|     M|1991-01-26|      47017|
| 10010|1963-06-01| Duangkaew| Piveteau|     F|1989-08-24|      80324|
| 10040|1959-09-13|     Weiyi|  Meriste|     F|1993-02-14|      72668|
| 10030|1958-07-14|     Elvis|  Demeyer|     M|1994-02-17|      88806|
| 10050|1958-05-21|   Yinghua|   Dredge|     M|1990-12-25|      97830|
+------+----------+----------+---------+------+----------+-----------+
only showing top 5 rows



In [ ]:
# Exercise 2: show all employees who were never managers
# 2.1 load dep_manager data
# 2.2 do anti join on employees and dept_manager table

In [32]:
# 2.1 load dep_manager data

managerDF = (
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("url", "jdbc:postgresql://postgres:5432/rtjvm")
    .option("user", "docker")
    .option("password", "docker")
    .option("dbtable", "public.dept_manager")
    .load()
)

managerDF.printSchema()

root
 |-- dept_no: string (nullable = true)
 |-- emp_no: integer (nullable = true)
 |-- from_date: date (nullable = true)
 |-- to_date: date (nullable = true)



In [34]:
# 2.2 do anti join on employees and dept_manager table

nonManagerEmployees = ( 
    employeesDF
    .join(managerDF, 
          employeesDF["emp_no"] == managerDF["emp_no"], 
          "left_anti")
)

nonManagerEmployees.show(5)

+------+----------+----------+---------+------+----------+
|emp_no|birth_date|first_name|last_name|gender| hire_date|
+------+----------+----------+---------+------+----------+
| 12940|1953-10-25|  Odinaldo|   Farrar|     F|1987-12-12|
| 13840|1954-11-13|     Remco|    Demke|     M|1992-06-09|
| 14450|1963-08-01|  Fumitaka|Prochazka|     F|1985-04-26|
| 14570|1963-07-26|    Chinho|     Bala|     F|1994-11-17|
| 15790|1960-03-20|     Kokou| Schnabel|     M|1991-04-03|
+------+----------+----------+---------+------+----------+
only showing top 5 rows



In [ ]:
#  3. find job titles of the top 10 best employees of the company,
#  max(toDate) is current job title


# 3.1 load titles dataframe
# 3.2 write sql
# 3.3 transformations - make title table be current title table then join with employees

In [36]:
# 3.1 load titles dataframe

titlesDF = (
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("user", "docker")
    .option("password", "docker")
    .option("url", "jdbc:postgresql://postgres:5432/rtjvm")
    .option("dbtable", "public.titles")
    .load()
)

titlesDF.printSchema()

root
 |-- emp_no: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- from_date: date (nullable = true)
 |-- to_date: date (nullable = true)



In [40]:
# 3.2 write sql

# this table has latest title for an employee, then can join this employee data

# WITH max_emp_date as (SELECT emp_no, max(to_date) as maxDate FROM public.titles
# group by emp_no)
# SELECT t.emp_no, t.title, m.maxDate FROM public.titles t 
# JOIN max_emp_date m ON t.emp_no = m.emp_no AND t.to_date = m.maxDate;

In [79]:
# step 3.3 - tranformation 1 - create table with employee max to date
from pyspark.sql.functions import max

maxEmpDateDF = (
    titlesDF
    .groupBy("emp_no")
    .agg(max("to_date").alias("max_to_date"))
)

maxEmpDateDF.show(5)

+------+-----------+
|emp_no|max_to_date|
+------+-----------+
| 12940| 9999-01-01|
| 13840| 1999-12-30|
| 14450| 1996-12-30|
| 14570| 9999-01-01|
| 15790| 9999-01-01|
+------+-----------+
only showing top 5 rows



In [68]:
# 3.4 - transform2 - join to get the title for latest max date

latestTitleDF = (
    titlesDF
    .join(maxEmpDateDF, (maxEmpDateDF["emp_no"] == titlesDF["emp_no"]) 
          & (titlesDF["to_date"] == maxEmpDateDF["max_to_date"]))
)

# remove duplicate emp_no
latestTitleDF = latestTitleDF.drop(maxEmpDateDF["emp_no"])

latestTitleDF.show(5)

+---------------+----------+----------+------+-----------+
|          title| from_date|   to_date|emp_no|max_to_date|
+---------------+----------+----------+------+-----------+
|       Engineer|1999-01-16|9999-01-01| 15390| 9999-01-01|
|Senior Engineer|1993-11-04|9999-01-01| 17060| 9999-01-01|
|   Senior Staff|1993-05-05|9999-01-01| 24740| 9999-01-01|
|   Senior Staff|2001-07-03|9999-01-01| 26020| 9999-01-01|
|       Engineer|1997-10-20|9999-01-01| 26910| 9999-01-01|
+---------------+----------+----------+------+-----------+
only showing top 5 rows



In [77]:
# 3.5 show example of how we go from one employee and many titles to only latest

print("Employee with several titles:")
titlesDF.filter(col("emp_no") == "10030").show()

print("Latest title for that same employee:")
latestTitleDF.filter(col("emp_no") == "10030").show()

Employee with several titles:
+------+---------------+----------+----------+
|emp_no|          title| from_date|   to_date|
+------+---------------+----------+----------+
| 10030|       Engineer|1994-02-17|2001-02-17|
| 10030|Senior Engineer|2001-02-17|9999-01-01|
+------+---------------+----------+----------+

Latest title for that same employee:
+---------------+----------+----------+------+-----------+
|          title| from_date|   to_date|emp_no|max_to_date|
+---------------+----------+----------+------+-----------+
|Senior Engineer|2001-02-17|9999-01-01| 10030| 9999-01-01|
+---------------+----------+----------+------+-----------+



In [73]:
# oin salary and job title
joinExpression = (salariesDF["emp_no"] == latestTitleDF["emp_no"]) & (salariesDF["to_date"] == latestTitleDF["to_date"])
titleToSalary = latestTitleDF.join(salariesDF, joinExpression)

# remove ambigous emp no
titleToSalary = titleToSalary.drop(salariesDF["emp_no"])

titleToSalary.show(5)

+---------------+----------+----------+------+-----------+------+----------+----------+
|          title| from_date|   to_date|emp_no|max_to_date|salary| from_date|   to_date|
+---------------+----------+----------+------+-----------+------+----------+----------+
|Senior Engineer|2001-02-17|9999-01-01| 10030| 9999-01-01| 88806|2002-02-15|9999-01-01|
|   Senior Staff|2001-09-28|9999-01-01| 10080| 9999-01-01| 72729|2001-09-26|9999-01-01|
|   Senior Staff|2000-12-06|9999-01-01| 10160| 9999-01-01|102165|2001-12-04|9999-01-01|
|   Senior Staff|2002-06-16|9999-01-01| 10230| 9999-01-01| 61667|2002-06-15|9999-01-01|
|          Staff|1995-01-27|9999-01-01| 10280| 9999-01-01| 80485|2002-01-25|9999-01-01|
+---------------+----------+----------+------+-----------+------+----------+----------+
only showing top 5 rows



In [76]:
# sort by salary and show top 10
from pyspark.sql.functions import desc


top10TitlesBySalary = (
    titleToSalary
    .select("emp_no", "title", "salary")
    .orderBy(desc("salary"))
)

top10TitlesBySalary.show(10)

+------+------------+------+
|emp_no|       title|salary|
+------+------------+------+
|205000|Senior Staff|153715|
|246120|Senior Staff|146292|
|257360|Senior Staff|144748|
|107140|Senior Staff|142506|
|282030|Senior Staff|142184|
|282370|Senior Staff|141488|
|447240|Senior Staff|140332|
| 72680|Senior Staff|140178|
|295800|Senior Staff|139773|
|296250|Senior Staff|138716|
+------+------------+------+
only showing top 10 rows

